<a href="https://colab.research.google.com/github/Tejaswimadastu/Deep_Learning/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Positional Encoding + Self Attention

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import TextVectorization,Embedding,MultiHeadAttention

### Input Sentence

In [2]:
sentence=["I Love Deep Learning"]
print(sentence)


['I Love Deep Learning']


### Tokenization

In [5]:
vectorizer = TextVectorization(
    output_mode="int",
    output_sequence_length=4
)

vectorizer.adapt(sentence)

tokens = vectorizer(sentence)

print("Vocabulary:")
print(vectorizer.get_vocabulary())

print("Tokens:")
print(tokens.numpy())

Vocabulary:
['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]
Tokens:
[[4 2 5 3]]


### Word Embeddings

In [6]:
embedding_dim = 8

embedding_layer = Embedding(
    input_dim=len(vectorizer.get_vocabulary()),
    output_dim=embedding_dim
)

word_embeddings = embedding_layer(tokens)

print("Word Embeddings:")
print(word_embeddings.numpy())

Word Embeddings:
[[[ 0.01795932 -0.01892811  0.04639399  0.04112725  0.00650819
   -0.00642895  0.03689469 -0.01917992]
  [ 0.0458364  -0.01589401  0.03132692 -0.02217932  0.00211368
   -0.02097805 -0.02582332  0.03474419]
  [-0.0094489   0.00493706  0.00891247  0.02491292 -0.04624265
   -0.01700337 -0.00651234  0.03555263]
  [ 0.01377708 -0.04702958 -0.0305056  -0.02098272 -0.00941556
    0.04534015  0.04658903 -0.04083496]]]


### Positional Encoding Function

In [7]:
def positional_encoding(max_position, d_model):

    positions = np.arange(max_position)[:, np.newaxis]
    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_rads = positions * angle_rates

    PE = np.zeros((max_position, d_model))

    PE[:, 0::2] = np.sin(angle_rads[:, 0::2])
    PE[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(PE, dtype=tf.float32)

### Generate Positional Encoding

In [8]:
PE=positional_encoding(4,embedding_dim)
print(PE.numpy())

[[ 0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00
   0.0000000e+00  1.0000000e+00  0.0000000e+00  1.0000000e+00]
 [ 8.4147096e-01  5.4030228e-01  9.9833414e-02  9.9500418e-01
   9.9998331e-03  9.9994999e-01  9.9999981e-04  9.9999952e-01]
 [ 9.0929741e-01 -4.1614684e-01  1.9866933e-01  9.8006660e-01
   1.9998666e-02  9.9980003e-01  1.9999987e-03  9.9999797e-01]
 [ 1.4112000e-01 -9.8999250e-01  2.9552022e-01  9.5533651e-01
   2.9995501e-02  9.9955004e-01  2.9999956e-03  9.9999553e-01]]


### Add Positional Encoding

In [9]:
position_aware_embeddings = word_embeddings + PE[tf.newaxis, :]

print("Position-aware Embeddings:")
print(position_aware_embeddings.numpy())

Position-aware Embeddings:
[[[ 0.01795932  0.9810719   0.04639399  1.0411272   0.00650819
    0.99357104  0.03689469  0.98082006]
  [ 0.88730735  0.5244083   0.13116033  0.9728249   0.01211351
    0.97897196 -0.02482332  1.0347437 ]
  [ 0.8998485  -0.4112098   0.20758179  1.0049795  -0.02624399
    0.98279667 -0.00451235  1.0355506 ]
  [ 0.15489708 -1.0370221   0.26501462  0.93435377  0.02057995
    1.0448902   0.04958902  0.95916057]]]


### Multi Head Attention

In [10]:
attention_layer=MultiHeadAttention(
    num_heads=1,
    key_dim=embedding_dim
)



### Apply Self Attention

In [12]:
attention_output = attention_layer(
    query=position_aware_embeddings,
    value=position_aware_embeddings,
    key=position_aware_embeddings
)

print(attention_output.shape)
print("Contextualized Embeddings:")
print(attention_output.numpy())

(1, 4, 8)
Contextualized Embeddings:
[[[-0.16679148 -0.64181066  0.03577468  0.18009901 -0.03552501
    0.02916405 -0.31531113  0.34827667]
  [-0.16538462 -0.6412777   0.03710498  0.18105522 -0.03672897
    0.02892905 -0.31323004  0.34822202]
  [-0.16748731 -0.6387484   0.03607181  0.17994112 -0.03331238
    0.03261168 -0.31112558  0.34417492]
  [-0.17109555 -0.6360176   0.03383675  0.1778769  -0.02823209
    0.03731899 -0.31003794  0.33922806]]]
